## Process Lab Result Data

In [ ]:
import os

import pandas as pd
import numpy as np
import re

from datetime import date

data_path = os.path.join("..", "data")
raw_data_path = os.path.join(data_path, 'raw_data')
processed_data_path = os.path.join(data_path, 'processed_data')

## Load Data

### Patients of Interest

In [ ]:
cols = ['master_person_id', 'inclusion_date', 'endpoint_date']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"))[cols]

del cols

inclusion_patients_df['inclusion_date'] = pd.to_datetime(inclusion_patients_df['inclusion_date']).dt.date
inclusion_patients_df['endpoint_date'] = pd.to_datetime(inclusion_patients_df['endpoint_date']).dt.date

### Medication Data

In [ ]:
medications_df = pd.read_csv(os.path.join(raw_data_path, "20251203_medication_search_results.csv"))

medications_df['drug_exposure_start_date'] = pd.to_datetime(medications_df['drug_exposure_start_date'])

### Date Breakdown Data

In [ ]:
dates_breakdown_df = pd.read_csv(os.path.join(data_path, "ckd_patients_datebreakdown.csv"))

dates_breakdown_df['inclusion_date'] = pd.to_datetime(dates_breakdown_df['inclusion_date']).dt.date
dates_breakdown_df['endpoint_date'] = pd.to_datetime(dates_breakdown_df['endpoint_date']).dt.date
dates_breakdown_df['start_date'] = pd.to_datetime(dates_breakdown_df['start_date']).dt.date
dates_breakdown_df['end_date'] = pd.to_datetime(dates_breakdown_df['end_date']).dt.date

## Refined Meds

### Meds Prescribed during Inclusion Period

In [ ]:
cols = ['master_person_id', 'drug_exposure_start_date', 'drug_classification', 'drug_name']

medications_refined_df = dates_breakdown_df.merge(medications_df[cols], how='right', on='master_person_id')

start_filter = (medications_refined_df['drug_exposure_start_date'] >= medications_refined_df['start_date'])
end_filter = (medications_refined_df['drug_exposure_start_date'] <= medications_refined_df['end_date'])

medications_refined_df = medications_refined_df[start_filter&end_filter].drop_duplicates().reset_index(drop=True)

medications_refined_df['grouping'] = medications_refined_df['grouping'].astype(int)
medications_refined_df['drug_classification'] = medications_refined_df['drug_classification'].combine_first(medications_refined_df['drug_name'])
medications_refined_df['drug_classification'] = medications_refined_df['drug_classification'].str.replace(' ', '_')

medications_refined_df['drug_classification_count'] = medications_refined_df['drug_classification']

medications_refined_df = medications_refined_df.drop(columns=['drug_name'])

del start_filter, end_filter

medications_refined_df.head()

In [ ]:
agg = {'drug_classification_count' : 'count'}
grouping_cols = ['master_person_id', 'grouping', 'drug_classification']

medications_agg_df = medications_refined_df.groupby(grouping_cols).agg(agg).reset_index().sort_values(by=grouping_cols)

del agg, grouping_cols

medications_agg_df = medications_agg_df.pivot_table(values='drug_classification_count',
                                                    index=['master_person_id', 'grouping'],
                                                    columns='drug_classification',
                                                    fill_value=0,
                                                    aggfunc='sum').reset_index()

medications_agg_df.columns = ['master_person_id', 'grouping', 'ARB', 'Ace_Inhibitors', 'Beta_Blocker', 'CCB', 'Clopidogrel', 'DPP4', 'Doxazosin', 'Entresto', 'Ezetimibe',
                              'GLP1', 'Gliclazide', 'Hydralazine', 'Insulin', 'Isosorbide', 'MRA', 'Metformin', 'Minoxidil', 'Moxonidine', 'SLGT2i', 'Statin']

medications_agg_df.head()

### Meds Prescribed Prior to Inclusion

In [ ]:
cols = ['master_person_id', 'drug_exposure_start_date', 'drug_classification', 'drug_name']

prior_inclusion_medications_df = dates_breakdown_df.merge(medications_df[cols], how='right', on='master_person_id')

inclusion_date_filter = (prior_inclusion_medications_df['drug_exposure_start_date'] < prior_inclusion_medications_df['inclusion_date'])

drop_cols = ['endpoint_date', 'grouping', 'start_date', 'end_date']

prior_inclusion_medications_df = prior_inclusion_medications_df[inclusion_date_filter].drop(columns=drop_cols).drop_duplicates().reset_index(drop=True)

prior_inclusion_medications_df['drug_classification'] = prior_inclusion_medications_df['drug_classification'].combine_first(prior_inclusion_medications_df['drug_name'])
prior_inclusion_medications_df['drug_classification'] = prior_inclusion_medications_df['drug_classification'].str.replace(' ', '_')

prior_inclusion_medications_df['drug_classification_count'] = prior_inclusion_medications_df['drug_classification']

prior_inclusion_medications_df = prior_inclusion_medications_df.drop(columns=['drug_name'])

del inclusion_date_filter

prior_inclusion_medications_df.head()

In [ ]:
agg = {'drug_classification_count' : 'count'}
grouping_cols = ['master_person_id', 'drug_classification']

prior_inclusion_medications_agg_df = prior_inclusion_medications_df.groupby(grouping_cols).agg(agg).reset_index().sort_values(by=grouping_cols)

prior_inclusion_medications_agg_df = prior_inclusion_medications_agg_df.pivot_table(values='drug_classification_count',
                                                                                    index=['master_person_id'],
                                                                                    columns='drug_classification',
                                                                                    fill_value=0,
                                                                                    aggfunc='sum').reset_index()

prior_inclusion_medications_agg_df.columns = ['master_person_id', 'priorARB', 'priorAce_Inhibitors', 'priorBeta_Blocker', 'priorCCB', 'priorClopidogrel', 'priorDPP4',
                                              'priorDoxazosin', 'priorEntresto', 'priorEzetimibe', 'priorGLP1', 'priorGliclazide', 'priorHydralazine', 'priorInsulin',
                                              'priorIsosorbide', 'priorMRA', 'priorMetformin', 'priorMinoxidil', 'priorMoxonidine', 'priorSLGT2i', 'priorStatin']

prior_inclusion_medications_agg_df.head()

### Export Medication Data

In [ ]:
# --- Save Results ---
file_name = "20260306_processed_medication_data.csv"

medications_agg_df.to_csv(f"{processed_data_path}/{file_name}", index=False)
print("✅ Results saved.")

In [ ]:
# --- Save Results ---
file_name = "20260416_prior_inclusion_medication_data.csv"

prior_inclusion_medications_agg_df.to_csv(f"{processed_data_path}/{file_name}", index=False)
print("✅ Results saved.")